<a href="https://colab.research.google.com/github/himanshusar123/-Machine-Learning-Quiz-Classification-or-Regression-/blob/main/Day_3_updated_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import csv
import os
import sqlite3

DB_NAME = "securebank.db"  # [cite: 10]
CSV_NAME = "Day3_SecureBank_Customer_Master.csv"  #


def initialize_and_migrate():
    """Automatically sets up the database and migrates data with direct console output."""
    print("=" * 60)
    print("      SECUREBANK AUTOMATED MIGRATION & VERIFICATION")
    print("=" * 60)

    # 1. Establish connection and create the table schema (Sprint 1)
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS Customers (
            CustomerID TEXT PRIMARY KEY,
            CustomerName TEXT NOT NULL,
            Branch TEXT NOT NULL,
            AccountType TEXT NOT NULL,
            Balance REAL NOT NULL,
            Status TEXT NOT NULL DEFAULT 'Active'
        )
    """
    )  # [cite: 10]
    conn.commit()
    print("[SUCCESS] Step 1: SQLite Database & Customers table initialized.")

    # 2. Verify file existence and run migration (Sprint 2)
    if not os.path.exists(CSV_NAME):
        print(
            f"[CRITICAL ERROR] The file '{CSV_NAME}' was not found in this folder."
        )
        print("Please place your CSV file in the exact same directory as this script.")
        conn.close()
        return

    try:
        with open(CSV_NAME, mode="r", encoding="utf-8") as file:
            csv_reader = csv.DictReader(file)

            # Standardize header names to remove leading/trailing spaces
            csv_reader.fieldnames = [
                field.strip() for field in csv_reader.fieldnames
            ]

            inserted = 0
            skipped = 0

            for row in csv_reader:
                customer_id = row["Customer ID"].strip()

                # Duplicate verification pass
                cursor.execute(
                    "SELECT 1 FROM Customers WHERE CustomerID = ?",
                    (customer_id,),
                )
                if cursor.fetchone():
                    skipped += 1
                    continue

                cursor.execute(
                    """
                    INSERT INTO Customers (CustomerID, CustomerName, Branch, AccountType, Balance, Status)
                    VALUES (?, ?, ?, ?, ?, ?)
                """,
                    (
                        customer_id,
                        row["Customer Name"].strip(),
                        row["Branch"].strip(),
                        row["Account Type"].strip(),
                        float(row["Balance"]),
                        row["Status"].strip(),
                    ),
                )
                inserted += 1

        conn.commit()
        print(
            f"[SUCCESS] Step 2: Migration Complete. Imported: {inserted}, Skipped (Duplicates): {skipped}"
        )

    except Exception as e:
        conn.rollback()
        print(f"[ERROR] Migration failed: {e}")
        conn.close()
        return

    # 3. Retrieve and visibly output rows immediately (Sprint 3)
    print("\n[VERIFICATION] Reading freshly written database rows:")
    print("-" * 75)
    print(
        f"{'ID':<7} | {'Name':<15} | {'Branch':<12} | {'Type':<10} | {'Balance':<10} | {'Status':<8}"
    )
    print("-" * 75)

    cursor.execute("SELECT * FROM Customers")
    rows = cursor.fetchall()

    for row in rows:
        print(
            f"{row[0]:<7} | {row[1]:<15} | {row[2]:<12} | {row[3]:<10} | {row[4]:<10,.2f} | {row[5]:<8}"
        )

    print("-" * 75)
    print(f"Total rows currently verified inside SQL database: {len(rows)}")
    conn.close()


if __name__ == "__main__":
    initialize_and_migrate()

      SECUREBANK AUTOMATED MIGRATION & VERIFICATION
[SUCCESS] Step 1: SQLite Database & Customers table initialized.
[CRITICAL ERROR] The file 'Day3_SecureBank_Customer_Master.csv' was not found in this folder.
Please place your CSV file in the exact same directory as this script.
